### objective
the goal of this notebook is to clean and prepare the nyc taxi dataset for fare prediction.

the notebook is designed to make the workflow easy to follow.
the idea is not only to transform the data, but also to explain why each step is necessary.

the main goals are:

- inspect the raw dataset
- detect missing or inconsistent observations
- clean the trip records
- create useful features for fare prediction
- remove variables that would create leakage or unnecessary redundancy

the final result will be a dataset ready for modeling `fare_amount`.

### load data

we dynamically find the project root so the notebook works from any folder.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

cwd = Path().resolve()
project_root = next(p for p in [cwd] + list(cwd.parents) if (p / "data").exists())

path_output = project_root / "data" / "procesed"
parquet_path = path_output / "master_2019_1M_per_month.parquet"

df = pd.read_parquet(parquet_path)


### first inspection

we check shape and structure to understand what we are working with.

In [ ]:
df.shape, df.info()

### missing values

we compute missing values ratio per column.

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing.head(25)

we notice that several variables have exactly the same percentage of missing values.

this usually means the same rows are missing multiple fields → not random missingness.

this suggests:
- possible ingestion issue
- or corrupted subset of rows

so we check if those rows are identical.

In [ ]:
core_vendor_cols = [
    "VendorID",
    "store_and_fwd_flag",
    "passenger_count"
]
mask_block_missing = df[core_vendor_cols].isna().all(axis=1)

print("Rows fully missing vendor block:",
      mask_block_missing.sum(),
      "| %:", mask_block_missing.mean())

the rows are exactly the same across all these variables.

decision:
- drop them
- reason: structural error + very small % of dataset

keeping them would introduce noise.

In [ ]:
df = df.loc[~mask_block_missing].copy()

### time variables

timestamps are critical because we need them to compute duration.

first we ensure they are properly parsed.

In [ ]:
for c in ["tpep_pickup_datetime","tpep_dropoff_datetime"]:
    if df[c].dtype == "object":
        df[c] = pd.to_datetime(df[c], errors="coerce")
        

In [ ]:
df = df.dropna(subset=["tpep_pickup_datetime","tpep_dropoff_datetime"])
df = df.drop(columns=["year"], errors="ignore")
df["month"] = df["tpep_pickup_datetime"].dt.month

we drop rows with invalid timestamps because we cannot compute trip duration.

we also drop "year" because it is redundant once datetime is available.

### duration and speed

we compute basic physical quantities:
- duration (minutes)
- speed (mph)

these help us detect unrealistic trips.

In [ ]:
df["duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["speed_mph"] = df["trip_distance"] / (df["duration_min"] / 60)

In [ ]:
df = df[
    (df["speed_mph"] > 0) &
    (df["speed_mph"] < 120) &
    (df["duration_min"] < 6*60)
].copy()

filters applied:

- duration > 0 → removes impossible trips
- duration < 6h → removes extreme outliers
- speed > 0 → removes invalid cases
- speed < 120 mph → unrealistic in city

idea: keep only physically plausible trips.

### check missing again

In [ ]:
df.isna().mean().sort_values(ascending=False).head(15)

### time features

we extract basic temporal structure.


these features capture behavior differences:
- hour → traffic cycles
- weekday → business vs leisure
- weekend → simplified indicator

In [ ]:
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_weekday"] = df["tpep_pickup_datetime"].dt.weekday
df["is_weekend"] = (df["pickup_weekday"] >= 5).astype(int)

### cyclical encoding

some time variables are circular.

for example, hour 23 and hour 0 are close in reality, even if they look far apart numerically.

to preserve this circular structure, we encode hour and month using sine and cosine transformations.

In [ ]:
df["hour_sin"] = np.sin(2*np.pi*df["pickup_hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["pickup_hour"]/24)

df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

this avoids artificial discontinuities in models.

this makes location interpretable and allows spatial features.

### distance transformations

distance is one of the main drivers of fare, but its relationship with price is not always perfectly linear.

to allow the model to capture nonlinearity, we create:

- `log_trip_distance`
- `trip_distance_sq`

In [ ]:
df["log_trip_distance"] = np.log1p(df["trip_distance"])
df["trip_distance_sq"] = df["trip_distance"] ** 2

this helps models capture curvature.

### traffic indicators

traffic conditions can affect fare because the ride may take place in more congested or less congested periods.

we create simple binary indicators for:

- rush hour
- night trips
- peak daytime

In [ ]:
df["is_rush_hour"] = df["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)
df["is_night"] = df["pickup_hour"].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)

df["is_peak_daytime"] = (
    (df["pickup_hour"] >= 11) &
    (df["pickup_hour"] <= 18)
).astype(int)

traffic conditions strongly affect trip dynamics.

### interaction features

sometimes the effect of distance depends on context.

for example, a long trip during rush hour may behave differently from a long trip on a quiet weekend.

to capture this, we create interaction terms.

In [ ]:
df["distance_x_rush"] = df["trip_distance"] * df["is_rush_hour"]
df["distance_x_weekend"] = df["trip_distance"] * df["is_weekend"]

these help capture context-dependent effects.

### final dataset for fare prediction

the target of the model is `fare_amount`.

because of this, we must remove variables that would create target leakage.

these include variables that directly contain monetary information derived from the fare, such as:

- total amount
- tip amount
- tolls
- surcharges
- airport fee

if these were kept, the model would appear highly accurate, but the result would not be meaningful.

In [ ]:
# %%
cols_to_drop = [
    # direct monetary leakage
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee",

    # raw datetime columns already converted into useful features
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",

    # weak or low-value variables for the first model
    "payment_type",
    "store_and_fwd_flag",


    # avoid duplication with cyclical encoding
    "pickup_hour",
    "month",

    # redundant with trip_distance and duration
    "speed_mph"
]

df_model = df.drop(columns=cols_to_drop, errors="ignore").copy()

### why some variables are left out

some variables are excluded on purpose.

`total_amount`, `tip_amount`, `tolls_amount`, and other surcharges are removed because they are direct monetary outcomes linked to the fare.
keeping them would create leakage.

`payment_type` is left out because it is not a strong ex-ante trip characteristic and may reflect behavior observed later in the process.

`store_and_fwd_flag` is excluded because it is operationally weak for this task and adds little economic interpretation.

`pickup_hour` and `month` are dropped because their cyclical versions already preserve the information in a better form.

`speed_mph` is removed because it is constructed from distance and duration, so it adds redundancy.

`PU_Zone`, `DO_Zone`, `PULocationID`, and `DOLocationID` are excluded from the first final model to reduce dimensionality and keep the specification more interpretable.

`distance_bin` is useful for exploration, but not essential once continuous distance transformations are already included.

### final check

we now inspect the final dataset that will be used for fare prediction.

In [ ]:
print("Final model shape:", df_model.shape)
print("\nFinal columns:\n")
print(sorted(df_model.columns.tolist()))

### conclusion

the dataset is now in a cleaner and more defensible form for predicting `fare_amount`.

the cleaning process followed four main ideas:

1. remove structurally problematic rows and invalid timestamps
2. keep only physically plausible trips
3. create variables that capture time, space, and trip structure
4. remove variables that would create leakage or add unnecessary redundancy

the final dataset keeps the variables that help explain fare behavior without directly using post-fare monetary outcomes.

this is important because the objective is not just to obtain a good prediction, but to build a model that is interpretable and methodologically correct.

in summary, the final dataset balances:
- data quality
- feature richness
- interpretability
- protection against leakage

this makes it a solid starting point for supervised learning on taxi fare prediction.